<a href="https://colab.research.google.com/github/quinterogilalejandro-wq/Actividad-15/blob/main/Actividad_15_Trabajo_Cooperativo_sobre_Data_Journey%2C_Acceso_y_Manipulaci%C3%B3n_de_Datos%2C_Monitoreo_y_Logging%2C_y_Model_Serving_con_Weights_%26_Biases.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# =============================================================================
#  CICLO DE VIDA DE UN MODELO DE DEEP LEARNING
#  Conceptos: Data Journey · Acceso/Manipulación de Datos ·
#             Monitoreo & Logging con W&B · Model Serving
# =============================================================================


import os
import numpy as np
import tensorflow as tf
import wandb
from wandb.integration.keras import WandbMetricsLogger, WandbModelCheckpoint
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns
from flask import Flask, request, jsonify

# Semilla global para reproducibilidad (trazabilidad de experimentos)
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)


# =============================================================================
# ETAPA 1 — DATA JOURNEY: INGESTA, TRANSFORMACIÓN Y PARTICIONAMIENTO
# =============================================================================
# El "Data Journey" describe el ciclo completo que recorren los datos:
#   Fuente → Ingesta → Limpieza → Transformación → Particionamiento → Modelo
#
# Una gestión cuidadosa de los datos garantiza que el modelo aprenda
# patrones reales y no artefactos del preprocesamiento.
# =============================================================================

print("=" * 65)
print("ETAPA 1 · DATA JOURNEY — Ingesta y Preparación de Datos")
print("=" * 65)

def etapa1_data_journey():
    """
    Implementa el flujo completo de datos sobre el dataset público MNIST
    (LeCun et al., 1998), cargado directamente desde tf.keras.datasets.

    MNIST es el benchmark estándar de reconocimiento de dígitos escritos
    a mano: 70 000 imágenes en escala de grises de 28×28 píxeles,
    particionadas oficialmente en 60 000 (train) y 10 000 (test),
    con 10 clases (dígitos del 0 al 9).

    Fuente pública:
      LeCun, Y., Cortes, C., & Burges, C. J. C. (1998).
      The MNIST database of handwritten digits.
      http://yann.lecun.com/exdb/mnist/

    Pasos del pipeline:
      1. Ingesta       — descarga automática desde tf.keras.datasets
      2. Exploración   — auditoría de forma, tipo y estadísticas descriptivas
      3. Aplanamiento  — reshape de (N, 28, 28) → (N, 784) para la MLP
      4. Limpieza      — detección de NaN, valores fuera de rango y duplicados
      5. Normalización — escalado Min-Max a [0, 1] para estabilidad del gradiente
      6. Codificación  — One-Hot Encoding de etiquetas para categorical_crossentropy
      7. Partición     — extracción de subconjunto de validación desde el split oficial
    """

    # ── 1. INGESTA DESDE FUENTE PÚBLICA ───────────────────────────────────────
    # tf.keras.datasets.mnist descarga los archivos binarios (.gz) de Yann LeCun
    # y los almacena en caché local (~/.keras/datasets/mnist.npz).
    # En ejecuciones posteriores se carga desde el caché sin conexión a internet.
    print("\n[1/7] Ingesta — Cargando MNIST desde tf.keras.datasets …")
    print("      Fuente: http://yann.lecun.com/exdb/mnist/ (caché local en ~/.keras)")
    (X_train_full, y_train_full), (X_test_raw, y_test_raw_orig) = \
        tf.keras.datasets.mnist.load_data()

    print(f"      Split oficial  → Train: {X_train_full.shape} | Test: {X_test_raw.shape}")
    print(f"      Tipo de datos  → X: {X_train_full.dtype} | y: {y_train_full.dtype}")
    print(f"      Rango de pixel → Mín: {X_train_full.min()} | Máx: {X_train_full.max()}")

    # ── 2. EXPLORACIÓN / AUDITORÍA ────────────────────────────────────────────
    print("\n[2/7] Exploración — Estadísticas descriptivas del dataset …")
    print(f"      Total muestras train   : {len(X_train_full):,}")
    print(f"      Total muestras test    : {len(X_test_raw):,}")
    print(f"      Dimensiones imagen     : 28 × 28 píxeles (escala de grises)")
    print(f"      Clases (dígitos)       : {sorted(np.unique(y_train_full).tolist())}")

    dist_clases = {int(c): int(np.sum(y_train_full == c)) for c in range(10)}
    print(f"      Distribución train     : {dist_clases}")
    print(f"      ¿Dataset balanceado?   : "
          f"{'Sí' if max(dist_clases.values()) - min(dist_clases.values()) < 1000 else 'No'}")

    media_global  = X_train_full.mean()
    std_global    = X_train_full.std()
    print(f"      Media de píxeles       : {media_global:.2f}")
    print(f"      Desv. estándar píxeles : {std_global:.2f}")

    # ── 3. APLANAMIENTO (RESHAPE) ─────────────────────────────────────────────
    # La MLP espera vectores 1D. Cada imagen 28×28 se convierte en un vector
    # de 784 elementos conservando el orden raster (fila por fila, izq → der).
    print("\n[3/7] Aplanamiento — Reshape (N, 28, 28) → (N, 784) …")
    X_train_flat = X_train_full.reshape(-1, 784).astype(np.float32)
    X_test_flat  = X_test_raw.reshape(-1, 784).astype(np.float32)
    print(f"      Shape resultante train : {X_train_flat.shape}")
    print(f"      Shape resultante test  : {X_test_flat.shape}")

    # ── 4. LIMPIEZA ───────────────────────────────────────────────────────────
    # Aunque MNIST es un dataset curado, se aplican verificaciones explícitas
    # de calidad que son obligatorias en pipelines de producción real.
    print("\n[4/7] Limpieza — Verificación de integridad del dataset …")

    # 4a. Detección de NaN
    nan_train = int(np.isnan(X_train_flat).sum())
    nan_test  = int(np.isnan(X_test_flat).sum())
    print(f"      NaN en train  : {nan_train}  | NaN en test  : {nan_test}")

    # 4b. Detección de valores fuera de rango [0, 255]
    fuera_rango = int(np.sum((X_train_flat < 0) | (X_train_flat > 255)))
    print(f"      Valores fuera de [0, 255] en train: {fuera_rango}")

    # 4c. Detección de muestras completamente negras (píxeles todos en 0)
    # Estas podrían ser imágenes corruptas o mal capturadas.
    negras = int(np.sum(X_train_flat.sum(axis=1) == 0))
    print(f"      Imágenes completamente negras (suma=0): {negras}")

    # 4d. Filtrado: conservar solo muestras válidas
    mascara_valida = (
        ~np.any(np.isnan(X_train_flat), axis=1) &
        np.all(X_train_flat >= 0, axis=1) &
        (X_train_flat.sum(axis=1) > 0)  # excluir imágenes totalmente negras
    )
    X_limpio = X_train_flat[mascara_valida]
    y_limpio = y_train_full[mascara_valida]
    print(f"      Muestras retenidas: {X_limpio.shape[0]:,} / {X_train_flat.shape[0]:,} "
          f"({mascara_valida.sum()/len(mascara_valida)*100:.2f}%)")

    # ── 5. NORMALIZACIÓN MIN-MAX ──────────────────────────────────────────────
    # Se dividen los valores entre 255 (máximo posible en uint8) para llevar
    # el rango de [0, 255] a [0.0, 1.0]. Esto:
    #   · Hace que los pesos iniciales de la red sean apropiados
    #   · Acelera la convergencia del optimizador Adam
    #   · Evita que neuronas con entradas grandes dominen el gradiente
    print("\n[5/7] Normalización — Escalado Min-Max → rango [0.0, 1.0] …")
    X_norm  = X_limpio / 255.0
    X_test_norm = X_test_flat / 255.0
    print(f"      Media post-normalización  : {X_norm.mean():.4f}  (era {media_global:.2f})")
    print(f"      Desv. std post-norm       : {X_norm.std():.4f}  (era {std_global:.2f})")
    print(f"      Rango [Mín, Máx]          : [{X_norm.min():.4f}, {X_norm.max():.4f}]")

    # ── 6. ONE-HOT ENCODING ───────────────────────────────────────────────────
    # La función de pérdida categorical_crossentropy requiere etiquetas en
    # formato one-hot: la clase 3 se representa como [0,0,0,1,0,0,0,0,0,0].
    print("\n[6/7] Codificación — One-Hot Encoding (10 clases) …")
    y_encoded      = tf.keras.utils.to_categorical(y_limpio, num_classes=10)
    y_test_encoded = tf.keras.utils.to_categorical(y_test_raw_orig, num_classes=10)
    print(f"      Ejemplo clase '5' → {y_encoded[y_limpio == 5][0].astype(int)}")
    print(f"      Shape y_train codificado : {y_encoded.shape}")

    # ── 7. PARTICIÓN TRAIN / VALIDACIÓN / TEST ────────────────────────────────
    # El set de test ya viene separado oficialmente en MNIST (10 000 muestras).
    # Del train oficial (60 000) se extrae un 15% como validación, garantizando
    # que el modelo NUNCA vea los datos de test durante el entrenamiento.
    print("\n[7/7] Particionamiento — Extrayendo validación del split oficial …")
    n        = X_norm.shape[0]
    n_val    = int(n * 0.15)
    n_train  = n - n_val

    # Shuffle reproducible antes de partir (evita sesgo de orden en el dataset)
    indices  = np.random.permutation(n)
    idx_tr   = indices[:n_train]
    idx_val  = indices[n_train:]

    X_train, y_train   = X_norm[idx_tr],          y_encoded[idx_tr]
    X_val,   y_val     = X_norm[idx_val],          y_encoded[idx_val]
    X_test,  y_test    = X_test_norm,              y_test_encoded
    y_test_raw         = y_test_raw_orig           # enteros originales para métricas

    print(f"      Train      : {X_train.shape[0]:,} muestras  ({n_train/n*100:.0f}%)")
    print(f"      Validación : {X_val.shape[0]:,} muestras  (15%)")
    print(f"      Test       : {X_test.shape[0]:,} muestras  (split oficial MNIST)")
    print("\n✔  Data Journey completado sobre dataset PÚBLICO MNIST.\n")

    return X_train, y_train, X_val, y_val, X_test, y_test, y_test_raw


X_train, y_train, X_val, y_val, X_test, y_test, y_test_raw = etapa1_data_journey()


# =============================================================================
# ETAPA 2 — MONITOREO Y LOGGING DEL ENTRENAMIENTO
# =============================================================================
# Esta etapa implementa un sistema de logging con MODO DUAL:
#
#   MODO A — Weights & Biases (W&B) [PREFERIDO]
#     Requiere: !pip install wandb  +  wandb.login()
#     Ventajas: dashboard web, comparación de experimentos, versionado de
#               artefactos, rastreo de gradientes, histogramas de pesos.
#
#   MODO B — Logging local [FALLBACK AUTOMÁTICO]
#     Se activa si W&B no está disponible o no hay autenticación.
#     Produce: registro CSV por época, 4 gráficas PNG, tabla de métricas
#              impresa en consola y reporte de clasificación completo.
#
# El código detecta automáticamente cuál modo usar. En cualquier caso,
# SIEMPRE se generan los archivos locales como evidencia tangible.
# =============================================================================

print("=" * 65)
print("ETAPA 2 · MONITOREO & LOGGING — W&B + Fallback Local")
print("=" * 65)

import io
import csv
from datetime import datetime

# ── HIPERPARÁMETROS CENTRALIZADOS ─────────────────────────────────────────────
config = {
    "proyecto"        : "ciclo-vida-deep-learning",
    "nombre_run"      : "experimento-mnist-mlp-v1",
    "learning_rate"   : 0.001,
    "epochs"          : 10,
    "batch_size"      : 32,
    "capas_ocultas"   : [256, 128, 64],
    "dropout_rate"    : 0.3,
    "activacion"      : "relu",
    "optimizador"     : "adam",
    "funcion_perdida" : "categorical_crossentropy",
    "num_clases"      : 10,
    "semilla"         : SEED,
    "dataset"         : "MNIST (LeCun et al., 1998) — tf.keras.datasets",
    "arquitectura"    : "MLP Fully-Connected con BatchNorm y Dropout"
}

# ── INTENTO DE CONEXIÓN A W&B ─────────────────────────────────────────────────
# Se intenta inicializar W&B. Si falla (sin cuenta, sin red, sin token),
# el sistema activa automáticamente el modo de logging local.
WANDB_ACTIVO = False
run = None

try:
    run = wandb.init(
        project = config["proyecto"],
        name    = config["nombre_run"],
        config  = config,
        tags    = ["deep-learning", "clasificacion", "mlp", "mnist-real"],
        notes   = (
            "Ciclo de vida completo: Data Journey sobre MNIST público, "
            "MLP con BatchNorm y Dropout, monitoreo dual W&B + local."
        )
    )
    WANDB_ACTIVO = True
    print(f"\n✔  [MODO A] Weights & Biases activo")
    print(f"   Run : {run.name}  |  ID: {run.id}")
    print(f"   URL : https://wandb.ai/{run.entity}/{config['proyecto']}")

except Exception as e:
    print(f"\n⚠  W&B no disponible ({type(e).__name__}). Activando MODO B — Logging Local.")
    print("   Todos los registros se guardarán en archivos locales.\n")

# ── CONSTRUCCIÓN DEL MODELO ───────────────────────────────────────────────────
# MLP (Multi-Layer Perceptron) con:
#   · BatchNormalization → normaliza activaciones entre capas, estabiliza el
#                          gradiente y permite tasas de aprendizaje más altas
#   · Dropout (30%)      → apaga aleatoriamente neuronas en cada paso de
#                          entrenamiento, forzando representaciones robustas
cfg = wandb.config if WANDB_ACTIVO else config

def construir_modelo(cfg):
    model = tf.keras.models.Sequential([
        tf.keras.layers.Input(shape=(784,), name="entrada"),

        # Bloque 1: extracción de características primarias
        tf.keras.layers.Dense(cfg["capas_ocultas"][0], name="densa_1"),
        tf.keras.layers.BatchNormalization(name="bn_1"),
        tf.keras.layers.Activation(cfg["activacion"], name="act_1"),
        tf.keras.layers.Dropout(cfg["dropout_rate"], name="dropout_1", seed=SEED),

        # Bloque 2: abstracción de nivel medio
        tf.keras.layers.Dense(cfg["capas_ocultas"][1], name="densa_2"),
        tf.keras.layers.BatchNormalization(name="bn_2"),
        tf.keras.layers.Activation(cfg["activacion"], name="act_2"),
        tf.keras.layers.Dropout(cfg["dropout_rate"], name="dropout_2", seed=SEED),

        # Bloque 3: representación comprimida
        tf.keras.layers.Dense(cfg["capas_ocultas"][2], name="densa_3"),
        tf.keras.layers.BatchNormalization(name="bn_3"),
        tf.keras.layers.Activation(cfg["activacion"], name="act_3"),

        # Salida: distribución de probabilidad sobre 10 clases
        tf.keras.layers.Dense(cfg["num_clases"], activation='softmax', name="salida")
    ], name="MLP_ClasificadorDL")

    model.compile(
        optimizer = tf.keras.optimizers.Adam(learning_rate=cfg["learning_rate"]),
        loss      = cfg["funcion_perdida"],
        metrics   = ['accuracy']
    )
    return model

model = construir_modelo(cfg)
model.summary()

# Registrar arquitectura en W&B si está disponible
if WANDB_ACTIVO:
    stream = io.StringIO()
    model.summary(print_fn=lambda x: stream.write(x + "\n"))
    wandb.log({"arquitectura_resumen": wandb.Html(f"<pre>{stream.getvalue()}</pre>")})
    # Nota: wandb.watch() solo es compatible con modelos PyTorch (torch.nn.Module).
    # Para Keras/TensorFlow el seguimiento de gradientes se realiza a través de
    # los callbacks WandbMetricsLogger y el callback personalizado MonitorDualCallback.
    print("\n✔  Arquitectura registrada en W&B.")

# ── LOGGER LOCAL (siempre activo como respaldo) ────────────────────────────────
# Registra cada época en un archivo CSV con marca de tiempo.
# Este archivo es la evidencia escrita del entrenamiento incluso sin W&B.
LOG_CSV = "registro_entrenamiento.csv"
_log_headers_escritos = False

def log_local_csv(datos: dict):
    """Agrega una fila al registro CSV local del experimento."""
    global _log_headers_escritos
    modo = "a" if _log_headers_escritos else "w"
    with open(LOG_CSV, modo, newline="") as f:
        writer = csv.DictWriter(f, fieldnames=datos.keys())
        if not _log_headers_escritos:
            writer.writeheader()
            _log_headers_escritos = True
        writer.writerow(datos)


# ── CALLBACKS DE ENTRENAMIENTO ─────────────────────────────────────────────────
class MonitorDualCallback(tf.keras.callbacks.Callback):
    """
    Callback unificado que envía métricas simultáneamente a:
      · W&B (si está activo): métricas estándar + métricas derivadas
      · CSV local (siempre): registro persistente por época

    Métricas derivadas registradas:
      · ratio_loss_val  — loss/val_loss; valores > 1.5 indican sobreajuste
      · gap_accuracy    — diferencia train_acc - val_acc; idealmente < 0.05
      · mejora_val_acc  — ganancia de accuracy de validación en esta época
    """
    def __init__(self):
        super().__init__()
        self._mejor_val_acc  = 0.0
        self._epoca_inicio   = datetime.now()

    def on_train_begin(self, logs=None):
        self._epoca_inicio = datetime.now()
        print(f"\n{'─'*60}")
        print(f"  Inicio del entrenamiento: {self._epoca_inicio.strftime('%H:%M:%S')}")
        print(f"  Hiperparámetros activos:")
        for k, v in config.items():
            if k not in ("proyecto", "nombre_run", "dataset", "arquitectura"):
                print(f"    {k:<20}: {v}")
        print(f"{'─'*60}\n")

    def on_epoch_end(self, epoch, logs=None):
        logs     = logs or {}
        loss     = logs.get("loss",         0.0)
        val_loss = logs.get("val_loss",     0.0)
        acc      = logs.get("accuracy",     0.0)
        val_acc  = logs.get("val_accuracy", 0.0)

        ratio    = round(loss / val_loss, 4) if val_loss > 0 else 0.0
        gap      = round(acc - val_acc,   4)
        mejora   = round(val_acc - self._mejor_val_acc, 4)

        # ── Registro CSV local (siempre) ──────────────────────────────────────
        log_local_csv({
            "timestamp"     : datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "epoca"         : epoch + 1,
            "loss"          : round(loss,     4),
            "accuracy"      : round(acc,      4),
            "val_loss"      : round(val_loss, 4),
            "val_accuracy"  : round(val_acc,  4),
            "ratio_loss_val": ratio,
            "gap_accuracy"  : gap,
            "mejora_val_acc": mejora,
        })

        # ── Logging a W&B (si está activo) ───────────────────────────────────
        if WANDB_ACTIVO:
            wandb.log({
                "ratio_loss_val" : ratio,
                "gap_accuracy"   : gap,
                "mejora_val_acc" : mejora,
            }, step=epoch)

        # ── Alerta temprana de sobreajuste en consola ─────────────────────────
        if ratio > 1.5:
            print(f"  ⚠  Época {epoch+1}: ratio loss/val_loss = {ratio:.2f} "
                  f"— posible sobreajuste")
        if val_acc > self._mejor_val_acc:
            self._mejor_val_acc = val_acc
            print(f"  ★  Época {epoch+1}: nuevo mejor val_accuracy = {val_acc:.4f}")

    def on_train_end(self, logs=None):
        duracion = (datetime.now() - self._epoca_inicio).seconds
        print(f"\n{'─'*60}")
        print(f"  Entrenamiento completado en {duracion}s")
        print(f"  Mejor val_accuracy alcanzado: {self._mejor_val_acc:.4f}")
        print(f"  Registro CSV guardado en: {LOG_CSV}")
        print(f"{'─'*60}")


# Construir lista de callbacks según el modo activo
callbacks = [MonitorDualCallback()]

if WANDB_ACTIVO:
    # Callbacks oficiales de W&B
    callbacks += [
        WandbMetricsLogger(log_freq="epoch"),
        WandbModelCheckpoint(
            filepath       = "mejor_modelo_wandb.keras",
            monitor        = "val_accuracy",
            save_best_only = True,
            verbose        = 0
        ),
    ]

# EarlyStopping y ModelCheckpoint local (activos siempre)
callbacks += [
    tf.keras.callbacks.EarlyStopping(
        monitor              = "val_loss",
        patience             = 3,
        restore_best_weights = True,
        verbose              = 1
    ),
    tf.keras.callbacks.ModelCheckpoint(
        filepath       = "mejor_modelo_local.keras",
        monitor        = "val_accuracy",
        save_best_only = True,
        verbose        = 0
    ),
]

# ── ENTRENAMIENTO ─────────────────────────────────────────────────────────────
print("\n" + "═" * 60)
print("  Iniciando entrenamiento del modelo …")
print("═" * 60)

historia = model.fit(
    X_train, y_train,
    validation_data = (X_val, y_val),
    epochs          = cfg["epochs"],
    batch_size      = cfg["batch_size"],
    callbacks       = callbacks,
    verbose         = 1
)

print("\n✔  Entrenamiento finalizado.\n")

# ── TABLA DE MÉTRICAS POR ÉPOCA (consola + W&B) ───────────────────────────────
# Esta tabla es la evidencia visual principal del logging.
# Se imprime siempre en consola y se registra como wandb.Table si W&B está activo.
print("─" * 65)
print(f"  {'Época':>5}  {'Loss':>8}  {'Acc':>8}  {'Val Loss':>9}  "
      f"{'Val Acc':>8}  {'Ratio':>6}  {'Gap':>6}")
print("─" * 65)

tabla_datos_wb = []
for ep in range(len(historia.history['loss'])):
    l    = historia.history['loss'][ep]
    a    = historia.history['accuracy'][ep]
    vl   = historia.history['val_loss'][ep]
    va   = historia.history['val_accuracy'][ep]
    rat  = round(l / vl, 3) if vl > 0 else 0.0
    gap  = round(a - va, 3)
    marca = " ★" if va == max(historia.history['val_accuracy']) else ""
    print(f"  {ep+1:>5}  {l:>8.4f}  {a:>8.4f}  {vl:>9.4f}  {va:>8.4f}  "
          f"{rat:>6.3f}  {gap:>6.3f}{marca}")
    tabla_datos_wb.append([ep+1, round(l,4), round(a,4), round(vl,4),
                           round(va,4), rat, gap])

print("─" * 65)
print("  ★ = mejor época por val_accuracy\n")

if WANDB_ACTIVO:
    wandb.log({
        "tabla_metricas_epocas": wandb.Table(
            columns=["Época","Loss","Accuracy","Val_Loss","Val_Accuracy",
                     "Ratio_Loss_Val","Gap_Accuracy"],
            data=tabla_datos_wb
        )
    })

# ── EVALUACIÓN FINAL SOBRE TEST ───────────────────────────────────────────────
print("─" * 65)
print("  Evaluación final — conjunto de TEST (nunca visto durante entrenamiento)")
print("─" * 65)

loss_test, acc_test = model.evaluate(X_test, y_test, verbose=0)
print(f"\n  Loss  (test) : {loss_test:.4f}")
print(f"  Acc   (test) : {acc_test:.4f}  ({acc_test*100:.2f}%)\n")

if WANDB_ACTIVO:
    wandb.log({"test_loss": loss_test, "test_accuracy": acc_test})

# ── PREDICCIONES Y MÉTRICAS DE CLASIFICACIÓN ─────────────────────────────────
y_pred_proba = model.predict(X_test, verbose=0)
y_pred       = np.argmax(y_pred_proba, axis=1)

reporte_txt  = classification_report(
    y_test_raw, y_pred,
    target_names=[f"Dígito {i}" for i in range(10)]
)
reporte_dict = classification_report(y_test_raw, y_pred, output_dict=True)

print("  Reporte de Clasificación por Clase:")
print("─" * 65)
print(reporte_txt)

# Guardar el reporte en un archivo de texto local
with open("reporte_clasificacion.txt", "w") as f:
    f.write(f"Experimento : {config['nombre_run']}\n")
    f.write(f"Dataset     : {config['dataset']}\n")
    f.write(f"Test Loss   : {loss_test:.4f}\n")
    f.write(f"Test Acc    : {acc_test:.4f}\n\n")
    f.write(reporte_txt)
print("  ✔  Reporte guardado en: reporte_clasificacion.txt")

if WANDB_ACTIVO:
    tabla_reporte_wb = wandb.Table(
        columns=["Clase","Precision","Recall","F1-Score","Support"],
        data=[
            [f"Dígito {k}", round(v["precision"],3), round(v["recall"],3),
             round(v["f1-score"],3), int(v["support"])]
            for k, v in reporte_dict.items() if k.isdigit()
        ]
    )
    wandb.log({"reporte_clasificacion": tabla_reporte_wb})

# ── GRÁFICA 1: CURVAS DE APRENDIZAJE ─────────────────────────────────────────
# Evidencia visual del comportamiento del modelo durante el entrenamiento.
# · Train vs Val Loss   → detecta sobreajuste (divergencia de curvas)
# · Train vs Val Acc    → mide generalización
epocas_eje = range(1, len(historia.history['loss']) + 1)
fig_lc, axes = plt.subplots(1, 2, figsize=(13, 5))
fig_lc.patch.set_facecolor('#F8F9FA')

for ax in axes:
    ax.set_facecolor('#FFFFFF')
    ax.grid(True, alpha=0.3, linestyle='--')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

axes[0].plot(epocas_eje, historia.history['loss'],     'o-', color='#2E5FA3',
             linewidth=2, markersize=5, label='Train Loss')
axes[0].plot(epocas_eje, historia.history['val_loss'], 's--', color='#E07B39',
             linewidth=2, markersize=5, label='Validación Loss')
axes[0].set_title('Curva de Pérdida (Loss)', fontweight='bold', fontsize=12)
axes[0].set_xlabel('Época'); axes[0].set_ylabel('Loss')
axes[0].legend(framealpha=0.9)

axes[1].plot(epocas_eje, historia.history['accuracy'],     'o-', color='#2E5FA3',
             linewidth=2, markersize=5, label='Train Accuracy')
axes[1].plot(epocas_eje, historia.history['val_accuracy'], 's--', color='#E07B39',
             linewidth=2, markersize=5, label='Validación Accuracy')
axes[1].set_title('Curva de Exactitud (Accuracy)', fontweight='bold', fontsize=12)
axes[1].set_xlabel('Época'); axes[1].set_ylabel('Accuracy')
axes[1].legend(framealpha=0.9)

plt.suptitle(
    f'Curvas de Aprendizaje — {config["nombre_run"]}  |  '
    f'Test Acc: {acc_test:.4f}',
    fontsize=13, fontweight='bold', y=1.02
)
plt.tight_layout()
plt.savefig("curvas_aprendizaje.png", dpi=150, bbox_inches='tight')
if WANDB_ACTIVO:
    wandb.log({"curvas_aprendizaje": wandb.Image(fig_lc)})
plt.close()
print("\n  ✔  Gráfica guardada: curvas_aprendizaje.png")

# ── GRÁFICA 2: MÉTRICAS DERIVADAS POR ÉPOCA ───────────────────────────────────
# Visualiza el ratio loss/val_loss y el gap de accuracy, dos indicadores
# clave para diagnosticar sobreajuste durante el entrenamiento.
epocas_hist = list(range(1, len(historia.history['loss']) + 1))
ratios = [
    round(l / vl, 4) if vl > 0 else 0
    for l, vl in zip(historia.history['loss'], historia.history['val_loss'])
]
gaps = [
    round(a - va, 4)
    for a, va in zip(historia.history['accuracy'], historia.history['val_accuracy'])
]

fig_der, ax2 = plt.subplots(1, 2, figsize=(13, 4))
fig_der.patch.set_facecolor('#F8F9FA')

ax2[0].plot(epocas_hist, ratios, 'o-', color='#C0392B', linewidth=2, markersize=5)
ax2[0].axhline(y=1.0, color='gray', linestyle='--', alpha=0.5, label='Ratio = 1.0 (ideal)')
ax2[0].axhline(y=1.5, color='#E07B39', linestyle=':', alpha=0.7, label='Umbral sobreajuste')
ax2[0].set_title('Ratio Loss / Val_Loss por Época', fontweight='bold')
ax2[0].set_xlabel('Época'); ax2[0].set_ylabel('Ratio')
ax2[0].legend(); ax2[0].grid(True, alpha=0.3); ax2[0].set_facecolor('#FFFFFF')

ax2[1].bar(epocas_hist, gaps, color=['#E07B39' if g > 0.05 else '#2E5FA3' for g in gaps],
           alpha=0.8, edgecolor='white')
ax2[1].axhline(y=0.05, color='gray', linestyle='--', alpha=0.5, label='Umbral gap (0.05)')
ax2[1].set_title('Gap de Accuracy (Train − Val) por Época', fontweight='bold')
ax2[1].set_xlabel('Época'); ax2[1].set_ylabel('Gap')
ax2[1].legend(); ax2[1].grid(True, alpha=0.3, axis='y'); ax2[1].set_facecolor('#FFFFFF')

plt.suptitle('Métricas de Diagnóstico de Sobreajuste', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig("metricas_diagnostico.png", dpi=150, bbox_inches='tight')
if WANDB_ACTIVO:
    wandb.log({"metricas_diagnostico": wandb.Image(fig_der)})
plt.close()
print("  ✔  Gráfica guardada: metricas_diagnostico.png")

# ── GRÁFICA 3: MATRIZ DE CONFUSIÓN ───────────────────────────────────────────
cm     = confusion_matrix(y_test_raw, y_pred)
cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100

fig_cm, ax_cm = plt.subplots(figsize=(9, 7))
fig_cm.patch.set_facecolor('#F8F9FA')
sns.heatmap(
    cm_pct, annot=True, fmt='.1f', cmap='Blues', ax=ax_cm,
    xticklabels=range(10), yticklabels=range(10),
    linewidths=0.5, linecolor='#DDDDDD', cbar_kws={"label": "% del total de la clase real"}
)
ax_cm.set_xlabel("Clase Predicha", fontsize=12, fontweight='bold')
ax_cm.set_ylabel("Clase Real",     fontsize=12, fontweight='bold')
ax_cm.set_title(
    f"Matriz de Confusión — Test MNIST\nTest Accuracy: {acc_test*100:.2f}%",
    fontsize=13, fontweight='bold', pad=15
)
plt.tight_layout()
plt.savefig("matriz_confusion.png", dpi=150, bbox_inches='tight')
if WANDB_ACTIVO:
    wandb.log({"confusion_matrix": wandb.Image(fig_cm, caption="Matriz de Confusión (%)")})
plt.close()
print("  ✔  Gráfica guardada: matriz_confusion.png")

# ── GRÁFICA 4: PRECISIÓN Y RECALL POR CLASE ──────────────────────────────────
clases     = [f"D{i}" for i in range(10)]
precisions = [reporte_dict[str(i)]["precision"] for i in range(10)]
recalls    = [reporte_dict[str(i)]["recall"]    for i in range(10)]
f1_scores  = [reporte_dict[str(i)]["f1-score"]  for i in range(10)]

x_pos  = np.arange(10)
ancho  = 0.28
fig_pr, ax_pr = plt.subplots(figsize=(12, 5))
fig_pr.patch.set_facecolor('#F8F9FA')
ax_pr.set_facecolor('#FFFFFF')

ax_pr.bar(x_pos - ancho, precisions, ancho, label='Precisión',  color='#2E5FA3', alpha=0.85)
ax_pr.bar(x_pos,          recalls,   ancho, label='Recall',     color='#E07B39', alpha=0.85)
ax_pr.bar(x_pos + ancho,  f1_scores, ancho, label='F1-Score',   color='#27AE60', alpha=0.85)
ax_pr.axhline(y=acc_test, color='#8E44AD', linestyle='--', linewidth=1.5,
              label=f'Test Accuracy global ({acc_test:.3f})')
ax_pr.set_xticks(x_pos); ax_pr.set_xticklabels([f'Dígito {i}' for i in range(10)])
ax_pr.set_ylim(0, 1.1); ax_pr.set_ylabel('Métrica'); ax_pr.set_xlabel('Clase')
ax_pr.set_title('Precisión, Recall y F1-Score por Clase — MNIST Test',
                fontweight='bold', fontsize=12)
ax_pr.legend(loc='lower right', framealpha=0.9)
ax_pr.grid(True, alpha=0.3, axis='y')
ax_pr.spines['top'].set_visible(False)
ax_pr.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig("precision_recall_clases.png", dpi=150, bbox_inches='tight')
if WANDB_ACTIVO:
    wandb.log({"precision_recall_por_clase": wandb.Image(fig_pr)})
plt.close()
print("  ✔  Gráfica guardada: precision_recall_clases.png")

# ── ARTEFACTO DEL MODELO EN W&B ───────────────────────────────────────────────
model.save("modelo_final.keras")
print("\n  ✔  Modelo guardado localmente: modelo_final.keras")

if WANDB_ACTIVO:
    artefacto = wandb.Artifact(
        name        = "modelo-mlp-mnist",
        type        = "model",
        description = "MLP entrenado sobre MNIST real con BatchNorm y Dropout.",
        metadata    = {
            "test_accuracy" : round(acc_test,   4),
            "test_loss"     : round(loss_test,  4),
            "epochs_run"    : len(historia.history['loss']),
            "dataset"       : "MNIST — tf.keras.datasets",
            "framework"     : "TensorFlow/Keras",
        }
    )
    artefacto.add_file("modelo_final.keras")
    run.log_artifact(artefacto)
    wandb.finish()
    print("  ✔  Artefacto registrado y versionado en W&B.")
    print("  ✔  Sesión W&B cerrada — todos los logs persistidos.\n")
else:
    print("\n  Archivos de evidencia generados (MODO B — Logging Local):")
    for archivo in [LOG_CSV, "reporte_clasificacion.txt",
                    "curvas_aprendizaje.png", "metricas_diagnostico.png",
                    "matriz_confusion.png", "precision_recall_clases.png"]:
        print(f"    · {archivo}")
    print()


# =============================================================================
# ETAPA 3 — GESTIÓN DE ARTEFACTOS Y MODEL SERVING (PUESTA EN OPERACIÓN)
# =============================================================================
# Model Serving es el proceso de exponer un modelo entrenado como un
# servicio consumible por otras aplicaciones.
#
# Opciones comunes de serving:
#   · REST API local    → Flask / FastAPI (desarrollo / prototipado)
#   · TF Serving        → Servidor de alta performance para TensorFlow
#   · Cloud endpoints   → AWS SageMaker, GCP Vertex AI, Azure ML
#   · Serverless        → AWS Lambda + API Gateway
#
# En este ejemplo se implementa una API REST con Flask que simula
# el comportamiento de un endpoint de inferencia en producción.
# =============================================================================

print("=" * 65)
print("ETAPA 3 · MODEL SERVING — API REST con Flask")
print("=" * 65)

# Persistencia del modelo en formatos de producción
os.makedirs("model", exist_ok=True)
model.save("model/modelo_final.keras")  # Formato nativo Keras 3 (recomendado)
model.export("model/saved_model_tf")       # SavedModel exportado para TF Serving
print("\nModelo guardado en dos formatos:")
print("  · model/modelo_final.keras  → Formato nativo Keras 3")
print("  · model/saved_model_tf/     → SavedModel para TF Serving")

# ── SERVIDOR DE INFERENCIA (FLASK) ────────────────────────────────────────────
app = Flask(__name__)

# Cargar el modelo UNA SOLA VEZ al iniciar el servidor.
# En producción, esta carga ocurre en el startup del contenedor/servicio.
_modelo_cargado = None

def obtener_modelo():
    """Lazy loading con caché: carga el modelo en memoria solo una vez."""
    global _modelo_cargado
    if _modelo_cargado is None:
        _modelo_cargado = tf.keras.models.load_model("model/modelo_final.keras")
        print("[Serving] Modelo cargado en memoria.")
    return _modelo_cargado


@app.route('/health', methods=['GET'])
def health_check():
    """
    Endpoint de verificación de disponibilidad del servicio.
    Los orquestadores (Kubernetes, ECS) llaman a este endpoint
    periódicamente para detectar fallos y reiniciar contenedores.
    """
    return jsonify({"status": "healthy", "modelo": "clasificador-dl-v1"})


@app.route('/predict', methods=['POST'])
def predict():
    """
    Endpoint principal de inferencia.

    Contrato de la API:
      Entrada (JSON):
        {
          "input": [[0.1, 0.2, ..., 0.9]]   ← lista de listas, shape (N, 784)
        }

      Salida (JSON):
        {
          "status"     : "success",
          "prediccion" : [3, 7, ...],        ← clase con mayor probabilidad
          "probabilidades": [[...], [...]]   ← vector softmax completo
        }
    """
    try:
        data = request.get_json(force=True)

        if "input" not in data:
            return jsonify({"status": "error",
                            "mensaje": "Campo 'input' requerido en el body JSON"}), 400

        entrada = np.array(data["input"], dtype=np.float32)

        # Validar dimensionalidad de la entrada
        if entrada.ndim == 1:
            entrada = entrada.reshape(1, -1)
        if entrada.shape[1] != 784:
            return jsonify({"status": "error",
                            "mensaje": f"Se esperan 784 features, se recibieron {entrada.shape[1]}"}), 400

        # Inferencia
        modelo = obtener_modelo()
        probabilidades = modelo.predict(entrada, verbose=0)
        clases         = np.argmax(probabilidades, axis=1).tolist()

        return jsonify({
            "status"         : "success",
            "prediccion"     : clases,
            "probabilidades" : probabilidades.tolist(),
            "num_muestras"   : len(clases)
        })

    except Exception as e:
        return jsonify({"status": "error", "mensaje": str(e)}), 500


@app.route('/model-info', methods=['GET'])
def model_info():
    """
    Endpoint de metadatos del modelo.
    Permite a los consumidores conocer la versión y capacidades del servicio.
    """
    return jsonify({
        "nombre"       : "clasificador-dl-v1",
        "version"      : "1.0.0",
        "framework"    : "TensorFlow/Keras",
        "arquitectura" : "MLP Fully-Connected",
        "num_clases"   : 10,
        "input_shape"  : [784],
        "test_accuracy": round(acc_test, 4)
    })


print("\n✔  API Flask definida con 3 endpoints:")
print("   GET  /health      → Estado del servicio")
print("   POST /predict     → Inferencia sobre nuevos datos")
print("   GET  /model-info  → Metadatos del modelo")
print("\nPara iniciar el servidor, descomentar el bloque siguiente:")
print("""
  # ─── ACTIVAR SERVIDOR ────────────────────────────────────────
  # Ejecución estándar (local):
  # if __name__ == '__main__':
  #     app.run(host='0.0.0.0', port=5000, debug=False)
  #
  # En Google Colab usar pyngrok para exponer el puerto:
  # !pip install pyngrok -q
  # from pyngrok import ngrok
  # ngrok.set_auth_token("TU_TOKEN_NGROK")
  # tunnel = ngrok.connect(5000)
  # print("URL pública:", tunnel.public_url)
  # app.run(port=5000)
  # ─────────────────────────────────────────────────────────────
""")

# =============================================================================
# RESUMEN FINAL DEL CICLO DE VIDA
# =============================================================================
print("=" * 65)
print("RESUMEN — CICLO DE VIDA DEL MODELO COMPLETADO")
print("=" * 65)
print(f"""
  Etapa 1 · Data Journey
    - Dataset:        MNIST público — 70 000 imágenes 28×28 (LeCun et al., 1998)
    - Fuente:         tf.keras.datasets (caché local ~/.keras)
    - Preprocesado:   reshape, limpieza, normalización [0,1], one-hot encoding
    - Partición:      ~51K train | ~9K val | 10K test (split oficial MNIST)

  Etapa 2 · Monitoreo (W&B)
    - Hiperparámetros registrados : ✔
    - Gradientes y pesos (watch)  : ✔
    - Métricas por época          : ✔
    - Visualizaciones (CM + curvas): ✔
    - Artefacto del modelo        : ✔
    - Test accuracy final         : {acc_test:.4f}

  Etapa 3 · Model Serving
    - Modelo exportado (SavedModel): ✔
    - API REST (Flask)             : ✔
    - Endpoints: /health · /predict · /model-info
""")

ETAPA 1 · DATA JOURNEY — Ingesta y Preparación de Datos

[1/7] Ingesta — Cargando MNIST desde tf.keras.datasets …
      Fuente: http://yann.lecun.com/exdb/mnist/ (caché local en ~/.keras)
      Split oficial  → Train: (60000, 28, 28) | Test: (10000, 28, 28)
      Tipo de datos  → X: uint8 | y: uint8
      Rango de pixel → Mín: 0 | Máx: 255

[2/7] Exploración — Estadísticas descriptivas del dataset …
      Total muestras train   : 60,000
      Total muestras test    : 10,000
      Dimensiones imagen     : 28 × 28 píxeles (escala de grises)
      Clases (dígitos)       : [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
      Distribución train     : {0: 5923, 1: 6742, 2: 5958, 3: 6131, 4: 5842, 5: 5421, 6: 5918, 7: 6265, 8: 5851, 9: 5949}
      ¿Dataset balanceado?   : No
      Media de píxeles       : 33.32
      Desv. estándar píxeles : 78.57

[3/7] Aplanamiento — Reshape (N, 28, 28) → (N, 784) …
      Shape resultante train : (60000, 784)
      Shape resultante test  : (10000, 784)

[4/7] Limpieza — Ve


✔  [MODO A] Weights & Biases activo
   Run : experimento-mnist-mlp-v1  |  ID: p6bltaji
   URL : https://wandb.ai//ciclo-vida-deep-learning


Model: "MLP_ClasificadorDL"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ densa_1 (Dense)                 │ (None, 256)            │       200,960 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bn_1 (BatchNormalization)       │ (None, 256)            │         1,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ act_1 (Activation)              │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ densa_2 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bn_2 (BatchNormalization)       │ (None, 128)            │           512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ act_2 (Activation)              │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ densa_3 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bn_3 (BatchNormalization)       │ (None, 64)             │           256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ act_3 (Activation)              │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ salida (Dense)                  │ (None, 10)             │           650 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 244,554 (955.29 KB)

 Trainable params: 243,658 (951.79 KB)

 Non-trainable params: 896 (3.50 KB)


✔  Arquitectura registrada en W&B.

════════════════════════════════════════════════════════════
  Iniciando entrenamiento del modelo …
════════════════════════════════════════════════════════════

────────────────────────────────────────────────────────────
  Inicio del entrenamiento: 17:38:01
  Hiperparámetros activos:
    learning_rate       : 0.001
    epochs              : 10
    batch_size          : 32
    capas_ocultas       : [256, 128, 64]
    dropout_rate        : 0.3
    activacion          : relu
    optimizador         : adam
    funcion_perdida     : categorical_crossentropy
    num_clases          : 10
    semilla             : 42
────────────────────────────────────────────────────────────

Epoch 1/10
1592/1594 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.8146 - loss: 0.6255  ⚠  Época 1: ratio loss/val_loss = 2.79 — posible sobreajuste
  ★  Época 1: nuevo mejor val_accuracy = 0.9569
1594/1594 ━━━━━━━━━━━━━━━━━━━━ 16s 8ms/step - accuracy: 0.8832 - loss: 0.3955 - val_

epoch/accuracy,▁▅▆▆▇▇▇███
epoch/epoch,▁▂▃▃▄▅▆▆▇█
epoch/learning_rate,▁▁▁▁▁▁▁▁▁▁
epoch/loss,█▄▃▂▂▂▂▁▁▁
epoch/val_accuracy,▁▄▅▆▇▆▇▇██
epoch/val_loss,█▅▄▃▂▂▂▂▁▁
test_accuracy,▁
test_loss,▁
epoch/accuracy,0.9758
epoch/epoch,9
epoch/learning_rate,0.001


  ✔  Artefacto registrado y versionado en W&B.
  ✔  Sesión W&B cerrada — todos los logs persistidos.

ETAPA 3 · MODEL SERVING — API REST con Flask
Saved artifact at 'model/saved_model_tf'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 784), dtype=tf.float32, name='entrada')
Output Type:
  TensorSpec(shape=(None, 10), dtype=tf.float32, name=None)
Captures:
  140197207185488: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140197207188944: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140197207190096: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140197207185104: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140197207190288: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140197207189904: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140197207186640: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140197207189712: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140197207187216